# Colab Deep Generator Training

Run this notebook in Google Colab with GPU enabled. Do not run heavy model training on local CPU-only or older hardware.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())
print('Running in Colab-like environment:', IN_COLAB)
Path('artifacts/models').mkdir(parents=True, exist_ok=True)

In [ ]:
# Run in Colab after cloning or uploading the repository.
# !pip install -e ".[deep]"

In [ ]:
from regime_lab import fit_deep_generator, fit_regime_model, load_market_data, load_strategy_returns, make_return_windows, save_generator

market = load_market_data('examples/sample_market_prices.csv')
returns = load_strategy_returns('examples/sample_strategy_returns.csv')
regimes = fit_regime_model(market, n_regimes=3, lookback=20, min_train=30)
window_data = make_return_windows(returns, regimes.regimes, window_length=20, stride=1)

generator = fit_deep_generator(window_data.windows, window_data.labels, epochs=5, seed=42, backend='torch')
samples = generator.sample(8, regime=1, seed=7)
samples.shape

In [ ]:
save_generator(generator, 'artifacts/models/deep_generator.pkl')
augmented = generator.augment(window_data.windows, window_data.labels, n_per_regime=50, seed=42)
print('Augmented windows:', augmented.windows.shape)
print('Saved artifacts/models/deep_generator.pkl')